In [125]:
import pandas as pd
import numpy as np

# Tune file ka naam vcf_test.vcf rakha hai, toh hum path wahi denge
vcf_path = './vcf_test.vcf'

# 1. Dynamically dhoondhna ki header (asli data) kis line par hai
with open(vcf_path, 'r') as file:
    for line_index, line in enumerate(file):
        if line.startswith('#CHROM'):
            header_line = line_index
            break

# 2. Pandas ko bolna ki theek usi line number se padhna shuru kare
vcf_df = pd.read_csv(vcf_path, sep='\t', skiprows=header_line, nrows=10000)

# Data check karte hain!
vcf_df

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
0,chr1,10020,rs123,A,G,50.5,PASS,DP=100;AF=0.01;GENE=BRCA1
1,chr1,10035,rs456,T,C,20.0,FAIL,DP=15;AF=0.05;GENE=TP53
2,chr2,20500,rs789,G,"A,T",99.9,PASS,"DP=450;AF=0.5,0.1;GENE=EGFR"
3,chr3,35000,.,C,T,.,PASS,DP=10;AF=0.02


In [126]:
# 1. Poore dataframe mein jahan bhi '.' hai, usko asli missing value (NaN) se replace kar do
vcf_df = vcf_df.replace('.', np.nan)

# 2. QUAL column ko text se Number (Float) mein convert kar do taaki aage math / filter lag sake
vcf_df['QUAL'] = pd.to_numeric(vcf_df['QUAL'])

# Ab table aur uska data type check karte hain
print(vcf_df.dtypes)
print("\n--- Cleaned Table ---")
vcf_df

#CHROM     object
POS         int64
ID         object
REF        object
ALT        object
QUAL      float64
FILTER     object
INFO       object
dtype: object

--- Cleaned Table ---


,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
0,chr1,10020,rs123,A,G,50.5,PASS,DP=100;AF=0.01;GENE=BRCA1
1,chr1,10035,rs456,T,C,20.0,FAIL,DP=15;AF=0.05;GENE=TP53
2,chr2,20500,rs789,G,"A,T",99.9,PASS,"DP=450;AF=0.5,0.1;GENE=EGFR"
3,chr3,35000,NaN,C,T,NaN,PASS,DP=10;AF=0.02


In [127]:
# Ek chhota sa function banate hain jo INFO string ko dictionary mein todega
def extract_info(info_text):
    # Agar data missing (NaN) hai, toh khali dictionary return karo
    if pd.isna(info_text):
        return {}
    
    info_dict = {}
    # 1. Pehle ';' se todo
    for item in str(info_text).split(';'):
        # 2. Phir '=' se todo
        if '=' in item:
            key, value = item.split('=')
            info_dict[key] = value
            
    return info_dict

# --- Asli Magic Yahan Hota Hai ---

# Step 1: Function ko INFO column ke har row par apply karo
info_dictionaries = vcf_df['INFO'].apply(extract_info)

# Step 2: Un dictionaries se ek nayi table bana lo (isme DP, AF, GENE alag columns ban jayenge)
info_df = pd.DataFrame(info_dictionaries.tolist())

# Step 3: Apni purani table aur is nayi table ko aas-paas jod (Concat) do
final_vcf = pd.concat([vcf_df, info_df], axis=1)

# Step 4: Purana kachre wala 'INFO' column delete kar do, uska kaam ho gaya
final_vcf = final_vcf.drop('INFO', axis=1)

# Final clean table dekhte hain!
final_vcf

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,DP,AF,GENE
0,chr1,10020,rs123,A,G,50.5,PASS,100,0.01,BRCA1
1,chr1,10035,rs456,T,C,20.0,FAIL,15,0.05,TP53
2,chr2,20500,rs789,G,"A,T",99.9,PASS,450,"0.5,0.1",EGFR
3,chr3,35000,NaN,C,T,NaN,PASS,10,0.02,NaN


In [128]:
# Condition 1: Gene 'BRCA1' hona chahiye
# Condition 2: QUAL score 30 se bada hona chahiye
# Dono conditions ko '&' (AND) se jodenge taaki dono match hon

filtered_variants = final_vcf.loc[(final_vcf['GENE'] == 'BRCA1') & (final_vcf['QUAL'] > 30)]

print("--- Filtered Results ---")
filtered_variants

--- Filtered Results ---


,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,DP,AF,GENE
0,chr1,10020,rs123,A,G,50.5,PASS,100,0.01,BRCA1


In [129]:
# 1. Pehle DP column ko text se Number mein convert karte hain
final_vcf['DP'] = pd.to_numeric(final_vcf['DP'])

# 2. Basic statistics calculate karte hain
avg_quality = final_vcf['QUAL'].mean()
max_depth = final_vcf['DP'].max()

print("--- Stats Report ---")
print(f"Average Quality Score: {avg_quality}")
print(f"Highest Sequencing Depth: {max_depth}")

--- Stats Report ---
Average Quality Score: 56.800000000000004
Highest Sequencing Depth: 450


In [130]:
import pandas as pd
import numpy as np
import os

# 1. INFO todne wala function (Unchanged, Logic is perfect)
def extract_info(info_text):
    if pd.isna(info_text): return {}
    info_dict = {}
    for item in str(info_text).split(';'):
        if '=' in item:
            key, value = item.split('=')
            info_dict[key] = value
    return info_dict

# 2. THE UPGRADED PIPELINE (With Error Handling & nrows)
def run_pralay_smasher(file_path, rows_to_read=None):
    print(f"Loading data from: {file_path}...")
    
    # Error Handling: Check if file exists
    if not os.path.exists(file_path):
        print(f"❌ Error: File not found at {file_path}. Please check the path.")
        return None
        
    header_line = None
    
    # Error Handling & Smart Header Detection
    try:
        with open(file_path, 'r') as file:
            for line_index, line in enumerate(file):
                if line.startswith('#CHROM'):
                    header_line = line_index
                    break
                    
        # Agar #CHROM nahi mila (Corrupted file)
        if header_line is None:
            raise ValueError("Corrupted VCF: '#CHROM' header line not found in the file.")
            
        # Proper Reading with Chunking support
        df = pd.read_csv(file_path, sep='\t', skiprows=header_line, nrows=rows_to_read)
        
    except Exception as e:
        print(f"❌ Failed to read the file: {e}")
        return None
    
    # Scrubbing
    df = df.replace('.', np.nan)
    if 'QUAL' in df.columns:
        df['QUAL'] = pd.to_numeric(df['QUAL'], errors='coerce')
    
    # INFO Parsing
    if 'INFO' in df.columns:
        info_df = pd.DataFrame(df['INFO'].apply(extract_info).tolist())
        df = pd.concat([df, info_df], axis=1).drop('INFO', axis=1)
    
    # Type conversion for DP safely
    if 'DP' in df.columns:
        df['DP'] = pd.to_numeric(df['DP'], errors='coerce')
        
    print("VCF Smashed Successfully! 💥\n")
    return df

In [131]:
# --- 1. RUNNING DUMMY DATA ---
print("--- DUMMY TEST ---")
dummy_result = run_pralay_smasher('./vcf_test.vcf')

print("\n--- CLINVAR STRESS TEST ---")
# --- 2. RUNNING REAL CLINVAR DATA ---
clinvar_result = run_pralay_smasher('./clinvar.vcf', rows_to_read=10000)

# --- 3. FILTERING DANGEROUS MUTATIONS ---
if clinvar_result is not None and 'CLNSIG' in clinvar_result.columns:
    # Pathogenic rows nikalna
    khatarnak_mutations = clinvar_result[clinvar_result['CLNSIG'].str.contains('Pathogenic', na=False)]
    
    # 🔥 TERA NAYA PRO-LEVEL DEFENSIVE CODE YAHAN HAI 🔥
    expected_cols = ['#CHROM', 'POS', 'GENEINFO', 'CLNDN', 'CLNSIG']
    safe_cols = [col for col in expected_cols if col in clinvar_result.columns]
    final_view = khatarnak_mutations[safe_cols]
    
    print(f"Found {len(final_view)} Pathogenic Mutations!\n")
    print(final_view.head())
    
    # Exporting safely
    final_view.to_csv('114_pathogenic_mutations.csv', index=False)
    print("\nFile Saved Successfully! Phase 1 Complete! 🚀")
else:
    print("CLNSIG column not found in this dataset.")

--- DUMMY TEST ---
Loading data from: ./vcf_test.vcf...
VCF Smashed Successfully! 💥


--- CLINVAR STRESS TEST ---
Loading data from: ./clinvar.vcf...


C:\Users\Sarthak Shukla\AppData\Local\Temp\ipykernel_9092\2607061881.py:46: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('.', np.nan)


VCF Smashed Successfully! 💥

Found 114 Pathogenic Mutations!

      #CHROM      POS                 GENEINFO  \
1026       1   943995            SAMD11:148398   
1739       1   976611  AGRN:375790|PERM1:84808   
1830       1  1013983               ISG15:9636   
1872       1  1014143               ISG15:9636   
1915       1  1014316               ISG15:9636   

                                                  CLNDN  \
1026                                       not_provided   
1739                   Congenital_myasthenic_syndrome_8   
1830  Mendelian_susceptibility_to_mycobacterial_dise...   
1872  Mendelian_susceptibility_to_mycobacterial_dise...   
1915  Mendelian_susceptibility_to_mycobacterial_dise...   

                            CLNSIG  
1026                    Pathogenic  
1739                    Pathogenic  
1830  Pathogenic/Likely_pathogenic  
1872                    Pathogenic  
1915                    Pathogenic  

File Saved Successfully! Phase 1 Complete! 🚀


In [132]:
# Sirf in 114 bimaari wali mutations ko CSV mein save karna
final_view.to_csv('pathogenic_mutationslineupto_10000.csv', index=False)
print("File Saved! VCF Pipeline Phase 1 Complete! ")

File Saved! VCF Pipeline Phase 1 Complete! 


In [133]:
# Result ko Excel-friendly CSV mein save karna (index=False se extra numbering nahi aati)
automated_result.to_csv('clean_variants.csv', index=False)
print("File successfully saved as CSV!")

File successfully saved as CSV!
